# Solvent Accessibility Score Analysis

This pipeline allows you to retreive solvent accessibility scores to understand the level of buriedness of your binding site.

This is useful to understand, for example, wether your predicted binder is blocking your binding site of interest.

In [ ]:
"""
Generalized SASA Pipeline for Protein-Ligand Complexes
=======================================================
Computes per-residue solvent accessible surface area (SASA) for any PDB
containing a protein and a ligand (small molecule, metal ion, cofactor, etc.).

Binding-site residues are auto-detected as protein residues within a
configurable distance cutoff of any HETATM ligand atom.

Requirements:
    pip install biopython pandas numpy seaborn matplotlib
"""

import os
import glob
import warnings
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from Bio.PDB import PDBParser
from Bio.PDB.SASA import ShrakeRupley

warnings.filterwarnings("ignore")

# ==============================================================================
# CONFIGURATION — edit these values for your dataset
# ==============================================================================

INPUT_DIR = "/path/to/your/pdb/files"  # Directory containing .pdb files

# Residue names to treat as the ligand (HETATM).
# Set to None to auto-detect ALL HETATM residues (excluding water HOH/WAT).
# Examples: ["LIG"], ["ZN"], ["HEM"], ["ATP"]
LIGAND_RESNAMES = None

# Distance cutoff (Å) to define a residue as part of the binding site
BINDING_CUTOFF_ANGSTROM = 3.0

# SASA probe sphere resolution (higher = more accurate, slower)
SASA_N_POINTS = 960

# Maximum number of PDBs to process (set to None for all)
MAX_PDBS = None

# Output directory for saved figures
OUTPUT_DIR = os.path.join(INPUT_DIR, "sasa_results")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==============================================================================
# Setup
# ==============================================================================

parser = PDBParser(QUIET=True)
sr = ShrakeRupley(n_points=SASA_N_POINTS)

WATER_RESNAMES = {"HOH", "WAT", "H2O", "DOD"}

# ==============================================================================
# Helper: identify ligand atoms in a model
# ==============================================================================

def get_ligand_atoms(model):
    """
    Return all ligand atoms from HETATM records.
    Excludes water molecules unless the user explicitly lists water as a ligand.
    If LIGAND_RESNAMES is set, only those residue names are included.
    """
    ligand_atoms = []
    for chain in model:
        for residue in chain:
            hetflag = residue.id[0]
            resname = residue.get_resname().strip()
            is_hetatm = hetflag.startswith("H_") or hetflag == "W"
            if not is_hetatm:
                continue
            if resname in WATER_RESNAMES and (LIGAND_RESNAMES is None or resname not in LIGAND_RESNAMES):
                continue
            if LIGAND_RESNAMES is not None and resname not in LIGAND_RESNAMES:
                continue
            ligand_atoms.extend(list(residue.get_atoms()))
    return ligand_atoms


# ==============================================================================
# Core SASA computation per PDB
# ==============================================================================

def compute_residue_sasa_for_pdb(pdb_path):
    """
    Parse a PDB, compute SASA for every residue, and flag binding-site residues.

    Returns a DataFrame with columns:
        pdb_id, chain, resseq, resname, sasa, is_binding, min_dist_to_ligand
    """
    pdb_id = os.path.splitext(os.path.basename(pdb_path))[0]

    try:
        struct = parser.get_structure(pdb_id, pdb_path)
    except Exception as e:
        print(f"  [PARSE FAIL] {pdb_id}: {e}")
        return pd.DataFrame()

    try:
        sr.compute(struct, level="R")
    except Exception as e:
        print(f"  [SASA FAIL] {pdb_id}: {e}")
        return pd.DataFrame()

    rows = []
    for model in struct:
        ligand_atoms = get_ligand_atoms(model)
        if not ligand_atoms:
            print(f"  [WARN] {pdb_id}: no ligand atoms found — binding flags will all be False")
        ligand_coords = np.array([a.coord for a in ligand_atoms]) if ligand_atoms else None

        for chain in model:
            for residue in chain:
                chain_id = chain.id
                resseq = residue.id[1]
                resname = residue.get_resname().strip()
                sasa = getattr(residue, "sasa", np.nan)

                # Compute minimum distance from any residue atom to any ligand atom
                min_dist = np.nan
                is_binding = False
                if ligand_coords is not None:
                    res_coords = np.array([a.coord for a in residue.get_atoms()])
                    if len(res_coords) > 0:
                        dists = np.linalg.norm(
                            res_coords[:, None, :] - ligand_coords[None, :, :], axis=-1
                        )
                        min_dist = float(dists.min())
                        # Only flag protein residues as binding site
                        # (excludes the ligand residue itself and other HETATM)
                        if residue.id[0] == " " and min_dist < BINDING_CUTOFF_ANGSTROM:
                            is_binding = True

                rows.append({
                    "pdb_id": pdb_id,
                    "chain": chain_id,
                    "resseq": resseq,
                    "resname": resname,
                    "sasa": sasa,
                    "is_binding": is_binding,
                    "min_dist_to_ligand": min_dist,
                    "is_hetatm": residue.id[0] != " ",
                })

    return pd.DataFrame(rows)


# ==============================================================================
# Batch processing
# ==============================================================================

all_pdbs = sorted(glob.glob(os.path.join(INPUT_DIR, "*.pdb")))
if MAX_PDBS is not None:
    all_pdbs = all_pdbs[:MAX_PDBS]

print(f"Found {len(all_pdbs)} PDB files in {INPUT_DIR}\n")

df_list = []
for pdb_path in all_pdbs:
    pdb_id = os.path.splitext(os.path.basename(pdb_path))[0]
    print(f"Processing {pdb_id} ...")
    df = compute_residue_sasa_for_pdb(pdb_path)
    if len(df) > 0:
        n_binding = df["is_binding"].sum()
        print(f"  ✓ {len(df)} residues | {n_binding} binding-site residues")
        df_list.append(df)
    else:
        print(f"  ✗ Empty result — skipped")

if not df_list:
    raise RuntimeError(
        "No data was collected. Check that INPUT_DIR contains valid PDB files "
        "with HETATM ligand records."
    )

sasa_df = pd.concat(df_list, ignore_index=True)

# Separate protein-only rows for most analyses
protein_df = sasa_df[~sasa_df["is_hetatm"]].dropna(subset=["sasa"])

print(f"\n{'='*60}")
print(f"Total residues (all):    {len(sasa_df)}")
print(f"Total protein residues:  {len(protein_df)}")
print(f"Binding-site residues:   {protein_df['is_binding'].sum()}")
print(f"{'='*60}\n")
print(protein_df[["pdb_id", "resname", "sasa", "is_binding", "min_dist_to_ligand"]].head(10))


# ==============================================================================
# Plots
# ==============================================================================

# Colour palette
PALETTE = {True: "#E05C5C", False: "#5C9BE0"}

# --------------------------------------------------------------------------
# Plot 1: Violin — binding vs non-binding residue SASA (protein only)
# --------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(6, 5))
sns.violinplot(
    data=protein_df,
    x="is_binding",
    y="sasa",
    palette=PALETTE,
    inner="box",
    cut=0,
    ax=ax,
)
ax.set_xticklabels(["Non-binding", "Binding site"])
ax.set_xlabel("")
ax.set_ylabel("Residue SASA (Å²)")
ax.set_title("Binding-site vs Non-binding Residue SASA")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "01_sasa_binding_vs_nonbinding.png"), dpi=150)
plt.show()

# --------------------------------------------------------------------------
# Plot 2: Boxplot — SASA by residue type for binding-site residues
# --------------------------------------------------------------------------
binding_df = protein_df[protein_df["is_binding"]].copy()

if len(binding_df) == 0:
    print("[SKIP] No binding-site residues found — check BINDING_CUTOFF_ANGSTROM or ligand detection.")
else:
    order = (
        binding_df.groupby("resname")["sasa"]
        .median()
        .sort_values(ascending=False)
        .index.tolist()
    )
    fig, ax = plt.subplots(figsize=(max(8, len(order) * 0.6), 5))
    sns.boxplot(data=binding_df, x="resname", y="sasa", order=order, color="#E05C5C", ax=ax)
    ax.set_xlabel("Residue type")
    ax.set_ylabel("SASA (Å²)")
    ax.set_title("Binding-site Residue SASA by Residue Type")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "02_binding_sasa_by_restype.png"), dpi=150)
    plt.show()

# --------------------------------------------------------------------------
# Plot 3: Scatter — distance to ligand vs SASA (protein residues only)
# --------------------------------------------------------------------------
nearby_df = protein_df[protein_df["min_dist_to_ligand"] < 12].copy()

if len(nearby_df) > 0:
    fig, ax = plt.subplots(figsize=(7, 5))
    sc = ax.scatter(
        nearby_df["min_dist_to_ligand"],
        nearby_df["sasa"],
        c=nearby_df["is_binding"].map({True: "#E05C5C", False: "#5C9BE0"}),
        alpha=0.5,
        edgecolors="none",
        s=20,
    )
    ax.axvline(BINDING_CUTOFF_ANGSTROM, color="grey", linestyle="--", linewidth=1,
               label=f"Binding cutoff ({BINDING_CUTOFF_ANGSTROM} Å)")
    ax.set_xlabel("Min distance to ligand (Å)")
    ax.set_ylabel("Residue SASA (Å²)")
    ax.set_title("Distance to Ligand vs SASA\n(protein residues within 12 Å)")
    ax.legend()
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor="#E05C5C", label="Binding site"),
        Patch(facecolor="#5C9BE0", label="Non-binding"),
    ]
    ax.legend(handles=legend_elements)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "03_dist_vs_sasa_scatter.png"), dpi=150)
    plt.show()

# --------------------------------------------------------------------------
# Plot 4: Buried vs exposed breakdown for binding-site residues
#         (buried = SASA < 25 Å², exposed = SASA >= 25 Å²)
# --------------------------------------------------------------------------
BURIED_THRESHOLD = 25.0  # Å² — commonly used threshold

if len(binding_df) > 0:
    binding_df = binding_df.copy()
    binding_df["exposure"] = binding_df["sasa"].apply(
        lambda s: "Buried (<25 Å²)" if s < BURIED_THRESHOLD else "Exposed (≥25 Å²)"
    )
    counts = binding_df["exposure"].value_counts()

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.pie(
        counts,
        labels=counts.index,
        autopct="%1.1f%%",
        colors=["#5C9BE0", "#E05C5C"],
        startangle=90,
        wedgeprops=dict(edgecolor="white", linewidth=1.5),
    )
    ax.set_title("Binding-site Residue Burial")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "04_binding_burial_pie.png"), dpi=150)
    plt.show()

# --------------------------------------------------------------------------
# Plot 5: Per-structure mean SASA of binding-site residues
# --------------------------------------------------------------------------
if len(binding_df) > 0 and binding_df["pdb_id"].nunique() > 1:
    per_struct = (
        binding_df.groupby("pdb_id")["sasa"]
        .mean()
        .reset_index()
        .rename(columns={"sasa": "mean_binding_sasa"})
        .sort_values("mean_binding_sasa", ascending=False)
    )

    fig, ax = plt.subplots(figsize=(max(8, len(per_struct) * 0.5), 5))
    sns.barplot(data=per_struct, x="pdb_id", y="mean_binding_sasa", color="#E05C5C", ax=ax)
    ax.set_xlabel("Structure")
    ax.set_ylabel("Mean binding-site SASA (Å²)")
    ax.set_title("Per-structure Mean SASA of Binding-site Residues")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "05_per_structure_binding_sasa.png"), dpi=150)
    plt.show()

# ==============================================================================
# Summary CSV export
# ==============================================================================
out_csv = os.path.join(OUTPUT_DIR, "sasa_results.csv")
protein_df.to_csv(out_csv, index=False)
print(f"\nResults saved to: {out_csv}")
print(f"Figures saved to: {OUTPUT_DIR}")